# Extending: pipes in series

This example builds a new application from scratch and shows the extension points:

* **your own unit systems**, including several quantities that share a dimension;
* **parameters** of different types (a dropdown and a number);
* **a custom column type** (a whole-number count of fittings);
* **reacting to changes** with `on_update`, and styling with CSS classes.

The physics is the Darcy–Weisbach pressure drop through pipe segments in series, carrying
the same flow $Q$:

$$V = \frac{Q}{\pi D^2/4}, \qquad \text{Re} = \frac{\rho V D}{\mu}, \qquad
\Delta p = \left(f\,\frac{L}{D} + n_\text{elbows} K\right) \frac{\rho V^2}{2}$$

with $f = 64/\text{Re}$ for laminar flow and the Swamee–Jain approximation of the
Colebrook equation otherwise.

In [ ]:
import ipywidgets as widgets
import numpy as np
from IPython.display import HTML, display

from ipyrowtable import (
    ChoiceColumn,
    InputColumn,
    LiveFigure,
    NodeColumn,
    NumberColumn,
    OutputColumn,
    RowTable,
    Unit,
    UnitSystem,
    fmt_sig,
    side_by_side,
)

## Unit systems

Your compute function works in one set of *base* units; SI here (m, Pa, m³/s, m/s). Each
`UnitSystem` says how to *display* each quantity: `display = base × scale + offset`.
Quantity names are yours to choose, so diameter can be shown in mm while length is in m.

In [ ]:
METRIC = UnitSystem("Metric", {
    "length": Unit("m"),
    "diameter": Unit("mm", 1e3),
    "pressure": Unit("kPa", 1e-3),
    "flow": Unit("m³/h", 3600.0),
    "velocity": Unit("m/s"),
})

US = UnitSystem("US", {
    "length": Unit("ft", 1 / 0.3048),
    "diameter": Unit("in", 1 / 0.0254),
    "pressure": Unit("psi", 1 / 6894.757),
    "flow": Unit("gpm", 15850.32),
    "velocity": Unit("ft/s", 1 / 0.3048),
})

## A custom column type

Subclass `InputColumn` and create a widget with a `value`. Values are stored as given, so
there is nothing to convert.

In [ ]:
class CountColumn(InputColumn):
    """A whole number per row, e.g. how many fittings a segment has."""

    width = "90px"

    def __init__(self, key, header=None, default=0, width=None):
        super().__init__(key, header, default, width)

    def create_widget(self, value, units):
        return widgets.BoundedIntText(value=value, min=0, max=1000,
                                      layout=widgets.Layout(width="100%"))

## The model

In [ ]:
ROUGHNESS = {  # absolute roughness, m
    "Commercial steel": 0.045e-3,
    "Galvanized iron": 0.15e-3,
    "Cast iron": 0.26e-3,
    "Copper / PVC": 0.0015e-3,
}
FLUIDS = {  # density kg/m³, viscosity Pa·s
    "Water, 20 °C": (998.2, 1.002e-3),
    "Water, 60 °C": (983.2, 0.467e-3),
    "Light oil": (870.0, 0.04),
}
K_ELBOW = 0.9  # loss coefficient of a standard 90° elbow


def friction_factor(Re, relative_roughness):
    Re = np.maximum(Re, 1e-12)
    turbulent = 0.25 / np.log10(relative_roughness / 3.7 + 5.74 / Re**0.9) ** 2
    return np.where(Re < 2300, 64 / Re, turbulent)


def pipe_flow(inputs):
    L = inputs.column("length").astype(float)
    D = inputs.column("diameter").astype(float)
    if np.any(D <= 0):
        raise ValueError("Every segment needs a diameter greater than zero.")
    roughness = np.array([ROUGHNESS[m] for m in inputs.column("material")])
    elbows = inputs.column("elbows").astype(float)
    rho, mu = FLUIDS[inputs.params["fluid"]]

    V = inputs.params["flow"] / (np.pi * D**2 / 4)
    Re = rho * V * D / mu
    f = friction_factor(Re, roughness / D)
    dp = (f * L / D + K_ELBOW * elbows) * rho * V**2 / 2
    p_inlet = inputs.edges["pressure"][0]
    return {
        "velocity": V,
        "Re": Re,
        "dp": dp,
        "pressure": p_inlet - np.concatenate(([0.0], np.cumsum(dp))),
    }

## The table

Per-system defaults (`{"Metric": 50, "US": 2}`) keep new rows at round numbers in whichever
units are showing. `{unit}` in a header is replaced by the column's current unit.

The table is shown right away, in the cell that creates it; the next cell adds a plot that
follows it.

In [ ]:
def summary(results):
    u = results.units
    total = results.display("pressure")[0] - results.display("pressure")[-1]
    return (f"Total pressure drop <b>{fmt_sig(total)} {u.label('pressure')}</b>, "
            f"outlet pressure {fmt_sig(results.display('pressure')[-1])} {u.label('pressure')}")


pipes = RowTable(
    columns=[
        ChoiceColumn("material", "Material", options=ROUGHNESS, width="170px"),
        NumberColumn("length", "Length ({unit})", quantity="length",
                     default={"Metric": 10, "US": 30}, min=0, width="90px"),
        NumberColumn("diameter", "Diameter ({unit})", quantity="diameter",
                     default={"Metric": 50, "US": 2}, min=0, width="100px"),
        CountColumn("elbows", "Elbows"),
        OutputColumn("velocity", "V ({unit})", quantity="velocity", fmt="{:.2f}", width="80px"),
        OutputColumn("Re", "Re", fmt="{:,.0f}", width="90px"),
        OutputColumn("dp", "Δp ({unit})", quantity="pressure", width="90px"),
        NodeColumn("pressure", "Pressure ({unit})", quantity="pressure", inputs="first",
                   first_default={"Metric": 400, "US": 60}, width="120px"),
    ],
    compute=pipe_flow,
    parameters=[
        ChoiceColumn("fluid", "Fluid", options=FLUIDS, width="150px"),
        NumberColumn("flow", "Flow rate ({unit})", quantity="flow",
                     default={"Metric": 20, "US": 90}, min=0, width="90px"),
    ],
    rows=[
        {"material": "Commercial steel", "length": 25, "diameter": 65, "elbows": 2},
        {"material": "Commercial steel", "length": 10, "diameter": 50, "elbows": 3},
        {"material": "Copper / PVC", "length": 6, "diameter": 40, "elbows": 1},
    ],
    unit_systems=[METRIC, US],
    leading_label="— inlet —",
    item_name="segment",
    summary=summary,
)
pipes

Try raising the flow rate, shrinking a diameter, or switching the fluid to light oil
(laminar flow: watch Re).

## Plot and a design check

`on_update(callback)` runs your code after every change (with `None` while the inputs are
invalid). Here it flags segments faster than 3 m/s, a common limit for water to keep noise
and erosion down. The plot and the check follow the table above as you edit it.

In [ ]:
def draw_pressure(fig, results):
    u = results.units
    ax = fig.add_subplot()
    distance = np.concatenate(([0.0], np.cumsum(results.display("length"))))
    ax.plot(distance, results.display("pressure"), marker="o")
    ax.set_xlabel(f"Distance from inlet ({u.label('length')})")
    ax.set_ylabel(f"Pressure ({u.label('pressure')})")
    ax.grid(alpha=0.3)


check = widgets.HTML()


def check_velocity(results):
    if results is None:
        check.value = ""
        return
    fast = [str(i + 1) for i, v in enumerate(results["velocity"]) if v > 3.0]
    check.value = (f"⚠️ Segment {', '.join(fast)} faster than 3 m/s" if fast
                   else "✓ All velocities at or below 3 m/s")


pipes.on_update(check_velocity)
plot = LiveFigure(pipes, draw=draw_pressure, size=(5, 3.2))
widgets.VBox([plot, check])

## Styling

Every part of the table carries a CSS class: `ipyrowtable-grid`, `ipyrowtable-add`,
`ipyrowtable-message`, and `ipyrowtable-<key>` on each column's cells.

In [ ]:
display(HTML("<style>.ipyrowtable-dp { font-weight: 600; }</style>"))

## Results in code

In [ ]:
r = pipes.results
print(f"Velocities: {np.round(r['velocity'], 2)} m/s")
print(f"Reynolds:   {np.round(r['Re']).astype(int)}")
print(f"Outlet:     {r['pressure'][-1] / 1000:.1f} kPa")